In [ ]:
# %%
# Importação das bibliotecas necessárias
# Manipulação de dados
import os
import glob
import ctypes
import site
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


import pywt  # Biblioteca para transformacoes wavelet
import keras_tuner as kt
# Pré-processamento
from sklearn.preprocessing import MinMaxScaler

import keras

from keras import backend as K


# Componentes do modelo Seq2Seq com Atenção
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dropout, Dense, Concatenate, TimeDistributed, Attention
from tensorflow.keras.layers import (
    Conv1D,
    BatchNormalization,
    Activation,
    Add,
    LayerNormalization,
    Flatten,
    Reshape,
    SpatialDropout1D,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Métricas de avaliação
from sklearn.metrics import mean_absolute_error, mean_squared_error


# Otimizar alocacao de memoria GPU - reduz fragmentacao
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # Reduce TensorFlow logging verbosity

# Garante runtime de GPU no Arch Linux (cuSOLVER e libs CUDA via wheel)
def configure_tensorflow_gpu_runtime():
    """Configura paths CUDA do venv e pre-carrega bibliotecas antes do import do TensorFlow."""
    current_ld = os.environ.get('LD_LIBRARY_PATH', '')
    candidate_dirs = []

    for sp in site.getsitepackages():
        candidate_dirs.extend(glob.glob(os.path.join(sp, 'nvidia', '*', 'lib')))

    if 'VIRTUAL_ENV' in os.environ:
        candidate_dirs.extend(
            glob.glob(
                os.path.join(
                    os.environ['VIRTUAL_ENV'],
                    'lib',
                    'python*',
                    'site-packages',
                    'nvidia',
                    '*',
                    'lib',
                )
            )
        )

    valid_dirs = []
    for lib_dir in candidate_dirs:
        if os.path.isdir(lib_dir) and lib_dir not in valid_dirs:
            valid_dirs.append(lib_dir)

    if valid_dirs:
        merged = ':'.join(valid_dirs)
        os.environ['LD_LIBRARY_PATH'] = f"{merged}:{current_ld}" if current_ld else merged

        # Pre-carrega libs CUDA para evitar falha de resolucao dinamica no kernel do VS Code
        preload_libs = [
            'libcudart.so.12',
            'libcublas.so.12',
            'libcudnn.so.9',
            'libcusolver.so.11',
        ]
        for lib_name in preload_libs:
            for lib_dir in valid_dirs:
                lib_path = os.path.join(lib_dir, lib_name)
                if os.path.exists(lib_path):
                    try:
                        ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
                    except OSError:
                        pass
                    break

    return valid_dirs

cuda_lib_dirs = configure_tensorflow_gpu_runtime()
print('CUDA lib dirs found:', len(cuda_lib_dirs))
from tcn import TCN

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('Python:', os.environ.get('VIRTUAL_ENV', 'sem VIRTUAL_ENV'))
print('GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
import sys
from pathlib import Path
from tensorflow.keras.utils import plot_model

import matplotlib.dates as mdates
import pandas as pd
import keras_tuner as kt

project_root = Path.cwd().resolve()
if not (project_root / "data" / "dataset.csv").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.s2s_lstm_bi_wrapper_attention_extract import S2SLSTMBidirectionalWrapperAttentionExtract
from src.utils import wavelet_denoising

dataset = pd.read_csv(project_root / "data" / "dataset.csv")
dataset["id"] = pd.to_datetime(dataset["id"], format="mixed")
dataset = dataset.sort_values("id").set_index("id")

train_ratio = 0.75
val_ratio = 0.20

train_end = int(len(dataset) * train_ratio)
val_end = train_end + int(len(dataset) * val_ratio)

train_df = dataset.iloc[:train_end].copy()
val_df = dataset.iloc[train_end:val_end].copy()
test_df = dataset.iloc[val_end:].copy()

input_steps = 72
output_steps = 36

wrapper = S2SLSTMBidirectionalWrapperAttentionExtract().prepare(
    train_df,
    val_df,
    input_steps=input_steps,
    output_steps=output_steps,
    target_col="ws100_wavelet",
    denoise=["ws100"],
)


hp = kt.HyperParameters()
model = wrapper.build(hp)
plot_model(wrapper.model, to_file='model_structure.png', show_trainable=True, show_shapes=True, show_layer_names=True, expand_nested=True, dpi=300)
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
    ModelCheckpoint('../models/best_model.h5.keras', monitor='val_loss', save_best_only=True)
]
history = wrapper.fit(epochs=50, batch_size=32, verbose=1, callbacks=callbacks, use_validation=True)

print("Rows:", len(dataset))
print("Train/val/test:", len(train_df), len(val_df), len(test_df))
print("Final training loss:", history.history["loss"][-1])

In [ ]:
import seaborn as sns
def plotar_atencao(modelo_atencao, amostra_encoder, amostra_decoder):
    """Gera o Heatmap (Mapa de Calor) da atenção."""
    # Extrai os pesos de atenção usando o modelo extrator
    pesos_atencao = modelo_atencao.predict([amostra_encoder, amostra_decoder])
    
    # Remove a dimensão do batch (tamanho 1) para plotar a matriz 2D
    matriz_atencao = pesos_atencao[0]
    
    plt.figure(figsize=(24, 8))
    sns.heatmap(matriz_atencao, cmap="viridis", annot=False)
    plt.title("Mapa de Calor de Atenção (Seq2Seq)")
    plt.xlabel("Passos de Tempo do Encoder (Entrada Histórica)")
    plt.ylabel("Passos de Tempo do Decoder (Predição Futura)")
    plt.xticks(ticks=np.arange(0, amostra_encoder.shape[1], 1), labels=np.arange(0, amostra_encoder.shape[1], 1))
    plt.yticks(rotation=0)
    plt.show()

def get_inputs(wrapper, input_data):
        _, input_data = wrapper.prepare_data(input_data,
                                       wrapper.input_steps,
                                       wrapper.output_steps,
                                       wrapper.target_col,
                                       scaler_target=wrapper.scaler_target,
                                       scaler_other=wrapper.scaler_other,
                                       create_sequences=False)
        
        encoder_input = input_data.to_numpy(copy=True).reshape(1, wrapper.input_steps, input_data.shape[1])
        decoder_input = np.zeros((1, wrapper.output_steps, 1))
            
        decoder_input[0, 0, 0] = encoder_input[0, -1, wrapper.target_col_index]
            
        for t in range(1, wrapper.output_steps):
            previous_pred = decoder_input[0, t-1, 0]
            decoder_input[0, t, 0] = previous_pred

        return encoder_input, decoder_input
        
encoder_input, decoder_input = get_inputs(wrapper, test_df[-wrapper.input_steps:])
plotar_atencao(wrapper.attention_model, encoder_input, decoder_input)